In [ ]:
import numpy as np
import pandas as pd
import json
from datetime import timedelta
import pycountry
from numpyro import distributions as dist
import arviz as az
from IPython.display import display, Markdown

from matplotlib import pyplot as plt
from matplotlib.pyplot import subplots

from emu_renewal.constants import DATA_PATH, DATE_FORMAT, RUN_DATA_DELAY
from emu_renewal.inputs import get_cont_of_country, get_country_pop
from emu_renewal.renew import MultiStrainModel
from emu_renewal.run import find_run_start_time, find_run_end_time, get_country_vars, get_alpha_info, get_delta_info, get_mobility_provider, run_calibration
from emu_renewal.targets import SharedDispTarget

In [ ]:
mob_source = "fb_singletile_mob"
display(Markdown(f"mobility approach: {mob_source}"))
country = "France"
display(Markdown(f"\n country: {country}"))

In [ ]:
# Build the model
countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
iso3 = pycountry.countries.lookup("France").alpha_3
continent = get_cont_of_country(iso3)
pop = get_country_pop(iso3)
data_start = find_run_start_time(pop, iso3)
end_time = find_run_end_time(iso3, mob_source)
run_start = data_start - timedelta(RUN_DATA_DELAY)
start_str = run_start.strftime(DATE_FORMAT)
end_str = data_start.strftime(DATE_FORMAT)
var_data = get_country_vars(iso3)
delta_var, delta_targ, delta_seed = get_delta_info(iso3, var_data, continent, end_time)
alpha_var, alpha_targ, alpha_seed = get_alpha_info(iso3, var_data, continent, end_time, delta_targ)
start_var = "eu"
var_names = [start_var] + alpha_var
seed_times = [] + alpha_seed
mob_provider = get_mobility_provider(iso3, mob_source)
model = MultiStrainModel(pop, run_start, end_time, var_names, seed_times, mob_provider, False)
times = model.epoch.number_to_datetime(pd.Series(model.model_times))

In [ ]:
display(mob_provider.mobility_series.plot(title="mobility time series"))

In [ ]:
non_vp_params = {
    "gen_mean": 4.0,
    "gen_sd": 2.0,
    "cdr": 0.4,
    "ifr": 0.008,
    "report_mean": 8.0,
    "report_sd": 3.0,
    "death_mean": 25.0,
    "death_sd": 3.0,
    "admit_mean": 15.0,
    "admit_sd": 2.0,
    "stay_mean": 7.0,
    "stay_sd": 1.0,
    "har": 0.035,
    "icu_admit_mean": 10.0,
    "icu_admit_sd": 1.0,
    "icu_stay_mean": 7.0,
    "icu_stay_sd": 1.0,
    "icuar": 0.02,
    "cross_immunity": 0.5,
    "seed_rates": np.array([-13.0, -13.0]),
    "relinfect": np.array([1.2]),
    "seed_offsets": np.array([50.0]),
    "vacc_protect_hosp": 0.0,
    "vacc_protect_death": 0.0,
    "mob_exp": 1.0,
    "shared_dispersion": 0.3,
    "beta": 1.75,
}

In [ ]:
thinning = 7
proc = np.random.normal(0.0, 0.05, 62)
base_params = {"proc": proc} | non_vp_params
results = model.renewal_func(**base_params)
indicators = ["weekly_deaths", "weekly_cases"]
outputs = {}
for ind in indicators:
    outputs[ind] = pd.Series(results[ind], index=times)[::thinning]
display(Markdown(r"\newpage"))
display(Markdown("indicators: " + ", ".join(indicators)))
display(Markdown(f"indicator thinning to one target every {thinning} days"))

In [ ]:
fig, axes = plt.subplots(1, len(indicators), figsize=[10, 5])
outputs["weekly_deaths"].plot(ax=axes[0], marker="o", linewidth=0.0, markersize=3.0)
outputs["weekly_cases"].plot(ax=axes[1], marker="o", linewidth=0.0, markersize=3.0)

In [ ]:
#| output: false
beta_prior = {"beta": dist.Uniform(0.5, 2.5)}
mob_exp_prior = {"mob_exp": dist.Uniform(0.0, 2.0)}
targets = {ind: SharedDispTarget(targ, weight=20) for ind, targ in outputs.items()}
calib, mcmc = run_calibration(model, non_vp_params | beta_prior | mob_exp_prior, targets, True, 5000)

In [ ]:
idata = az.from_numpyro(mcmc)
n_samples = 20
proc_vals = idata.posterior["proc"].stack(sample=("chain", "draw")).values
beta_samples = idata.posterior["beta"].stack(sample=("chain", "draw")).values
mob_exp_samples = idata.posterior["mob_exp"].stack(sample=("chain", "draw")).values
samples = np.random.choice(proc_vals.shape[1], size=n_samples, replace=False)
sampled_procs = proc_vals[:, samples]

In [ ]:
# Rerun to get results for target
results = {ind: pd.DataFrame(index=times) for ind in indicators}
for i in range(n_samples):
    run_params = non_vp_params | {"proc": sampled_procs[:, i], "beta": beta_samples[i], "mob_exp": mob_exp_samples[i]}
    rerun_result = model.renewal_func(**run_params)
    for ind in indicators:
        results[ind][i] = rerun_result[ind]

# Plot fit
fig, axes = plt.subplots(1, len(indicators), figsize=[10, 5])
fig.suptitle("calibration fit")
for i, ind in enumerate(indicators):
    ax = axes[i]
    ax.set_title(ind)
    vals = results[ind]
    output = outputs[ind]
    for i in range(n_samples):
        ax.plot(times, vals[i], color="k", linewidth=0.2)
    ax.plot(output.index, output, marker="o", linewidth=0.0, markersize=3.0)
    ax.tick_params("x", rotation=70)

In [ ]:
display(Markdown(r"\newpage"))
display(Markdown(f"beta input: {non_vp_params['beta']}"))
display(Markdown(f"mob_exp input: {non_vp_params['mob_exp']}"))
display(Markdown("approximately recovers values of key parameters"))
fig = az.plot_trace(idata)
plt.tight_layout()

In [ ]:
# Process values
cum_proc = np.cumsum(sampled_procs, axis=0)
proc_result = pd.DataFrame(cum_proc, index=times[::7])

# Target
target_proc = np.concatenate([np.array([0.0]), proc])
cum_targ = np.cumsum(proc, axis=0)

In [ ]:
display(Markdown("recovers general trajectory of variable process"))
fig, ax = plt.subplots(1, 1)
fig.suptitle("recovery of variable process")
for i in range(n_samples):
    ax.plot(proc_result.index, proc_result[i], color="k", linewidth=0.2)
ax.plot(proc_result.index, cum_targ, marker="o", linewidth=0.0)

In [ ]:
display(Markdown("doesn't recover individual updates to variable process"))
fig, axes = subplots(9, 7, figsize=[20, 16], sharex=True)
flat_axes = axes.ravel()
plot_width = 0.1
for i in range(len(proc)):
    ax = flat_axes[i]
    fig = az.plot_posterior(idata.posterior["proc"][:, :, i], ax=ax, hdi_prob="hide", point_estimate=None)
    fig.vlines(proc[i], 0.0, 15.0, color="k")
    ax.set_title("")
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_xlim([-plot_width, plot_width])
flat_axes[-1].remove()
fig = ax.get_figure()
fig.tight_layout()